# LiveVQA Continual Baseline: Google Colab T4 Runner

This notebook runs the 4-bit QLoRA baseline on a free or standard Google Colab T4 GPU (16 GB VRAM).
It mounts Google Drive, sets up dependencies, executes training and evaluation, and preserves results.

## 1. Verify GPU Hardware

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Destination folder for persistent logs
!mkdir -p /content/drive/MyDrive/livevqa_experiments

## 3. Clone Repository & Install Dependencies

In [ ]:
# Replace with your repository URL or copy your local workspace
!git clone https://github.com/<YOUR_GITHUB_USERNAME>/livevqa-continual-baseline.git
%cd livevqa-continual-baseline

# Install core requirements and GPU-specific packages (bitsandbytes for 4-bit QLoRA, qwen-vl-utils)
!pip install -q -r requirements.txt
!pip install -q bitsandbytes>=0.43.0 qwen-vl-utils>=0.0.8

## 4. Prepare Dataset
Ensure you have placed the LiveVQA annotations in `data/livevqa_train.jsonl` and images in `data/images/` as described in `data/README.md`.

In [ ]:
# If dataset is stored on Drive, copy it over:
# !cp -r /content/drive/MyDrive/livevqa_data/* data/

# Verify data schema and report majority-class baseline
!python -m src.data --stats data/livevqa_train.jsonl

## 5. Execute Colab T4 Training (4-bit QLoRA, fp16)

In [ ]:
# Runs Qwen2.5-VL-3B-Instruct with 4-bit QLoRA, gradient accumulation 8 (effective batch 8)
!python -m src.train --config configs/colab_t4.yaml

## 6. Evaluate Seen vs Unseen & General Benchmark (MMStar)

In [ ]:
# Evaluates on split-by-image seen and unseen sets and audits raw predictions
!python -m src.eval --config configs/colab_t4.yaml

## 7. Run Continual Learning Benchmark (Optional Phase 2)

In [ ]:
# Evaluates sequential updates across K=3 chunks with experience replay
!python -m src.continual --config configs/colab_t4.yaml --num-chunks 3 --replay-size 16

## 8. Generate Analysis, Plots, and Sync Back to Google Drive

In [ ]:
!python -m src.analyze

# Sync all results and plots back to Google Drive
!cp -r results /content/drive/MyDrive/livevqa_experiments/
!cp -r checkpoints /content/drive/MyDrive/livevqa_experiments/
print("Execution complete! Results backed up to Google Drive.")